# Fase 1 — Ingeniería de datos y Zero Data Leakage

Construcción reproducible del dataset maestro ENAHO–ESCALE 2025. Las llaves se preservan como texto, los indicadores ESCALE se agregan por distrito y la población final se restringe a estudiantes de **6 a 17 años**.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import config
from src.data_loader import cargar_todas_las_bases
from src.feature_engineering import construir_dataset_maestro, exportar_dataset_maestro
print('Semilla global:', config.RANDOM_STATE)

Semilla global: 42


## 1. Ingesta segura

La carga maneja UTF-8/Latin-1, espacios en blanco como nulos y conserva `CONGLOME`, `VIVIENDA`, `HOGAR`, `CODPERSO` y `UBIGEO` como cadenas para evitar la pérdida de ceros a la izquierda.

In [2]:
bases = cargar_todas_las_bases()
pd.DataFrame({k: {'filas': len(v), 'columnas': v.shape[1]} for k, v in bases.items()}).T

,filas,columnas
m100,44599,9
m200,115145,7
m300,106958,16
sumaria,33702,10
padron,112724,9
padlocal,69642,3


## 2. Cruce relacional, población objetivo y variables derivadas

Se aplican validaciones `one_to_one`, `many_to_one` y agregaciones distritales de matrícula, docentes, gestión, ruralidad y locales escolares. El rezago se define como `edad > edad_normativa + 1` únicamente para primaria y secundaria con matrícula o asistencia vigente.

In [3]:
df = construir_dataset_maestro(bases)
exportar_dataset_maestro(df)
print('Dimensión final:', df.shape)
print('Rango de edad:', int(df[config.COL_EDAD].min()), '-', int(df[config.COL_EDAD].max()))

[INFO] M300 activas 2025: 106,958 de 106,958


[INFO] Población escolar 6-17 años: 23,668 de 106,958 registros
[INFO] Dataset maestro: 23,668 filas x 50 columnas


[OK] Exportado: C:\Users\Jalea\Documents\U\C8\inteligencia_artificial\u2_ex\proyecto_ia\data\processed\dataset_enaho_escale_2025.parquet
[OK] Exportado: C:\Users\Jalea\Documents\U\C8\inteligencia_artificial\u2_ex\proyecto_ia\data\processed\dataset_enaho_escale_2025.csv
[OK] Exportado: C:\Users\Jalea\Documents\U\C8\inteligencia_artificial\u2_ex\proyecto_ia\data\processed\diccionario_variables.csv
Dimensión final: (23668, 50)
Rango de edad: 6 - 17


## 3. Controles de calidad y protocolo anti-fuga

Los modelos posteriores usan `GroupShuffleSplit` y `GroupKFold` por `conglome_25`. Imputación, escalamiento y one-hot se ajustan dentro de cada fold. Las variables que construyen directamente el target (`p208a_25`, `p308a_25`, `p308b_25`, `edad_normativa`) se excluyen de la predicción.

In [4]:
assert df[config.COL_EDAD].between(config.EDAD_MIN, config.EDAD_MAX).all()
assert df[config.KEY_UBIGEO_ENAHO].dropna().str.len().eq(6).all()
assert df[config.GRUPO_CONGLOME].notna().all()
resumen = pd.Series({
    'registros_6_17': len(df),
    'distritos': df[config.KEY_UBIGEO_ENAHO].nunique(),
    'conglomerados': df[config.GRUPO_CONGLOME].nunique(),
    'evaluables_rezago': df[config.TARGET_REZAGO].notna().sum(),
    'casos_rezago': df[config.TARGET_REZAGO].eq(1).sum(),
    'prevalencia_rezago': df[config.TARGET_REZAGO].mean(),
})
display(resumen.to_frame('valor'))

,valor
registros_6_17,23668.000000
distritos,1240.000000
conglomerados,5050.000000
evaluables_rezago,9382.000000
casos_rezago,520.000000
prevalencia_rezago,0.055425


## Resultado

El dataset maestro, su versión CSV y el diccionario de variables quedan exportados en `data/processed/`. La población y las llaves cumplen los controles necesarios para las fases de clustering, clasificación y XAI.